In [0]:
import os
import subprocess
from html import escape
from typing import TypeAlias
from typing import List, Tuple, Union

from IPython.display import HTML, display
import subprocess

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

EnvironmentItem = Union[str, Tuple[str, str]]

def display_envs(
    name: str,
    variables: list[EnvironmentItem],
    sort: bool = True,
    reverse: bool = False,
    command_timeout: int = 30,
    title_color: str = "#2563eb",
) -> None:
    def get_variable_name(item: EnvironmentItem) -> str:
        return item if isinstance(item, str) else item[0]

    items = (
        sorted(
            variables,
            key=lambda item: get_variable_name(item).lower(),
            reverse=reverse,
        )
        if sort
        else variables
    )

    rows: list[str] = []

    for item in items:
        if isinstance(item, str):
            variable_name = item
            command = None
        else:
            variable_name, command = item

        environment_value = os.environ.get(variable_name)

        if environment_value is None:
            formatted_value = "<em>Not set</em>"
        else:
            formatted_value = (
                "<code style='white-space: pre-wrap; overflow-wrap: anywhere;'>"
                f"{escape(environment_value)}"
                "</code>"
            )

        formatted_command = ""
        formatted_result = ""

        if command:
            formatted_command = (
                "<code style='white-space: pre-wrap;'>"
                f"{escape(command)}"
                "</code>"
            )

            try:
                result = subprocess.run(
                    command,
                    shell=True,
                    executable="/bin/bash",
                    capture_output=True,
                    text=True,
                    timeout=command_timeout,
                    env=os.environ.copy(),
                )

                output_parts: list[str] = []

                if result.stdout.strip():
                    output_parts.append(result.stdout.rstrip())

                if result.stderr.strip():
                    output_parts.append(
                        f"stderr:\n{result.stderr.rstrip()}"
                    )

                if result.returncode != 0:
                    output_parts.append(
                        f"exit code: {result.returncode}"
                    )

                command_result = "\n\n".join(output_parts) or "(no output)"

            except subprocess.TimeoutExpired:
                command_result = (
                    f"Command timed out after {command_timeout} seconds"
                )

            except Exception as exception:
                command_result = (
                    f"{type(exception).__name__}: {exception}"
                )

            formatted_result = (
                "<pre style='"
                "margin: 0;"
                "white-space: pre-wrap;"
                "overflow-wrap: anywhere;"
                "font-size: 12px;"
                "'>"
                f"{escape(command_result)}"
                "</pre>"
            )

        rows.append(
            "<tr>"
            f"<td><strong>{escape(variable_name)}</strong></td>"
            f"<td>{formatted_value}</td>"
            f"<td>{formatted_command}</td>"
            f"<td>{formatted_result}</td>"
            "</tr>"
        )

    html = f"""
    <h2 style="
        color: {escape(title_color)};
        margin-bottom: 14px;
    ">
        {escape(name)}
    </h2>

    <table style="
        width: 100%;
        border-collapse: collapse;
        table-layout: fixed;
    ">
        <thead>
            <tr>
                <th style="width: 20%; text-align: left;">Variable</th>
                <th style="width: 30%; text-align: left;">Value</th>
                <th style="width: 20%; text-align: left;">Command</th>
                <th style="width: 30%; text-align: left;">Command result</th>
            </tr>
        </thead>

        <tbody>
            {''.join(rows)}
        </tbody>
    </table>

    <style>
        table th,
        table td {{
            border: 1px solid #ddd;
            padding: 8px;
            vertical-align: top;
            overflow-wrap: anywhere;
        }}
    </style>
    """

    display(HTML(html))

In [0]:
import subprocess

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


command_result_schema = ArrayType(
    StructType([
        StructField("command", StringType(), nullable=False),
        StructField("exit_code", IntegerType(), nullable=True),
        StructField("output", StringType(), nullable=True),
    ])
)


@F.udf(returnType=command_result_schema)
def run_commands_udf2(commands: list[str]):
    results = []

    for command in commands:
        try:
            process = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=10,
            )

            output = (
                process.stdout.strip()
                or process.stderr.strip()
                or ""
            )

            results.append({
                "command": command,
                "exit_code": process.returncode,
                "output": output,
            })

        except Exception as exc:
            results.append({
                "command": command,
                "exit_code": None,
                "output": f"{type(exc).__name__}: {exc}",
            })

    return results

def run_worker_commands2(commands: list[str]):
    commands_column = F.array(
        *[F.lit(command) for command in commands]
    )

    return (
        spark.range(1)
        .select(
            F.explode(
                run_commands_udf2(commands_column)
            ).alias("result")
        )
        .select(
            F.col("result.command").alias("command"),
            F.col("result.exit_code").alias("exit_code"),
            F.col("result.output").alias("output"),
        )
    )  